### Rename table column names to standard column name format

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import *  
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("landing_to_bronze")

catalog='catalog_dev'
source_schema='silver'
target_schema='gold'
storage_account='onprimtocloudstorgaeacc'

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {source_schema}")


DataFrame[]

In [0]:
# function to create dataframe

def create_df(table):
    logger.info(f"Starting ingestion for table: {catalog}.{source_schema}.{table} ")
    df=spark.table(table)
    return df

In [0]:
# function to modify column names to standard format

def modify_names(df):
    modified_columns=[]
    for column in df.columns:
        lst=list(column)
        index=0
        while index < len(lst):
            if index > 0 :
                if lst[index].isupper() and lst[index-1].islower():
                    lst.insert(index, '_')
                    index=index+2
                else:
                    index=index+1
            else:
                index=index+1
        modified_column=("".join(lst)).lower()
        logger.info(f"Modified {column} to {modified_column} ")
        df=df.withColumnRenamed(column, modified_column)
    return df


In [0]:
# function to save dataframe as table in Unity Catalog

def save_to_gold(df, table):
    target_path=f"abfss://{target_schema}@{storage_account}.dfs.core.windows.net/{table}"
    logger.info(f"Saving {table}")
    df.write.format('delta')\
        .mode('overwrite')\
        .option('path',target_path) \
        .option("overwriteSchema", "true")\
        .saveAsTable(f'{catalog}.{target_schema}.{table}')


In [0]:
def main():  
    matrix=spark.sql('''SELECT table_name
                    FROM catalog_dev.information_schema.tables
                    WHERE table_schema="silver"''')

    tables=[row.table_name  for row in matrix.collect()]
    #modified_names =[]
    for table in tables:
        df=create_df(table)
        modified_names_df=modify_names(df)
        save_to_gold(modified_names_df, table)  

    logger.info(f"Modification completed")
    logger.info(f"_______________________________________________________________________________________")

In [0]:
if __name__ == "__main__":
    main()

2026-06-03 13:59:49,835 - INFO - Starting ingestion for table: catalog_dev.silver.customeraddress 
2026-06-03 13:59:53,738 - INFO - Modified CustomerID to customer_id 
2026-06-03 13:59:53,739 - INFO - Modified AddressID to address_id 
2026-06-03 13:59:53,739 - INFO - Modified AddressType to address_type 
2026-06-03 13:59:53,740 - INFO - Modified rowguid to rowguid 
2026-06-03 13:59:53,740 - INFO - Modified ModifiedDate to modified_date 
2026-06-03 13:59:53,741 - INFO - Saving customeraddress
2026-06-03 14:00:08,480 - INFO - Starting ingestion for table: catalog_dev.silver.product 
2026-06-03 14:00:09,680 - INFO - Modified ProductID to product_id 
2026-06-03 14:00:09,681 - INFO - Modified Name to name 
2026-06-03 14:00:09,683 - INFO - Modified ProductNumber to product_number 
2026-06-03 14:00:09,684 - INFO - Modified Color to color 
2026-06-03 14:00:09,685 - INFO - Modified StandardCost to standard_cost 
2026-06-03 14:00:09,686 - INFO - Modified ListPrice to list_price 
2026-06-03 14:00